# 🤖 Lab 2: Seu Primeiro Modelo de Machine Learning

## Previsão de Churn de Clientes

### Objetivos de Aprendizagem:
- ✅ Entender o problema de classificação
- ✅ Preparar dados para Machine Learning
- ✅ Treinar um modelo preditivo
- ✅ Avaliar performance do modelo
- ✅ Fazer previsões em novos casos

### Duração: 35 minutos

---

## 📌 Contexto do Problema

Você trabalha em uma **empresa de telecomunicações** e precisa identificar quais clientes têm maior probabilidade de **cancelar o serviço** (churn).

**Por que isso é importante?**
- 💰 Custa 5x mais adquirir um novo cliente do que reter um existente
- 🎯 Podemos criar campanhas de retenção direcionadas
- 📊 Entender padrões de comportamento

**O que é Churn?**
- Cliente que cancela/abandona o serviço

## 📚 Passo 1: Importar Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Configurações
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
%matplotlib inline
np.random.seed(42)

print("✅ Bibliotecas importadas com sucesso!")

## 📊 Passo 2: Criar Dataset de Clientes

In [ ]:
# Gerar dataset simulado de 1000 clientes
n_clientes = 1000

dados_clientes = {
    'Tempo_Cliente_Meses': np.random.randint(1, 72, n_clientes),
    'Idade': np.random.randint(18, 70, n_clientes),
    'Gasto_Mensal': np.random.uniform(30, 200, n_clientes),
    'Num_Servicos': np.random.randint(1, 6, n_clientes),
    'Chamadas_Suporte': np.random.randint(0, 10, n_clientes),
    'Tipo_Contrato': np.random.choice(['Mensal', 'Anual', 'Bienal'], n_clientes, p=[0.5, 0.3, 0.2])
}

df = pd.DataFrame(dados_clientes)

# Criar target (Churn) com lógica realista
# Probabilidade maior de churn se:
# - Pouco tempo como cliente
# - Muitas chamadas para suporte
# - Contrato mensal
# - Baixo gasto mensal

prob_churn = (
    (df['Tempo_Cliente_Meses'] < 12) * 0.3 +
    (df['Chamadas_Suporte'] > 5) * 0.4 +
    (df['Tipo_Contrato'] == 'Mensal') * 0.25 +
    (df['Gasto_Mensal'] < 80) * 0.2 +
    np.random.uniform(0, 0.15, n_clientes)  # Ruído aleatório
)

df['Churn'] = (prob_churn > 0.6).astype(int)

print(f"✅ Dataset criado com {len(df)} clientes")
print(f"📊 Taxa de Churn: {df['Churn'].mean()*100:.1f}%")
print(f"   - Clientes ativos: {(df['Churn']==0).sum()}")
print(f"   - Clientes que cancelaram: {(df['Churn']==1).sum()}")

## 👀 Passo 3: Explorar os Dados

In [ ]:
# Visualizar primeiras linhas
print("📋 Primeiros 10 clientes:")
display(df.head(10))

In [ ]:
# Estatísticas por grupo (Churn vs Não Churn)
print("📊 Comparação: Clientes Ativos vs Churn\n")
print(df.groupby('Churn').agg({
    'Tempo_Cliente_Meses': 'mean',
    'Idade': 'mean',
    'Gasto_Mensal': 'mean',
    'Num_Servicos': 'mean',
    'Chamadas_Suporte': 'mean'
}).round(2))

In [ ]:
# Visualizar diferenças
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Tempo como cliente
df.boxplot(column='Tempo_Cliente_Meses', by='Churn', ax=axes[0,0])
axes[0,0].set_title('Tempo como Cliente')
axes[0,0].set_xlabel('Churn (0=Não, 1=Sim)')

# Gasto Mensal
df.boxplot(column='Gasto_Mensal', by='Churn', ax=axes[0,1])
axes[0,1].set_title('Gasto Mensal')
axes[0,1].set_xlabel('Churn (0=Não, 1=Sim)')

# Chamadas Suporte
df.boxplot(column='Chamadas_Suporte', by='Churn', ax=axes[1,0])
axes[1,0].set_title('Chamadas ao Suporte')
axes[1,0].set_xlabel('Churn (0=Não, 1=Sim)')

# Tipo de Contrato
pd.crosstab(df['Tipo_Contrato'], df['Churn'], normalize='index').plot(kind='bar', ax=axes[1,1])
axes[1,1].set_title('Churn por Tipo de Contrato')
axes[1,1].set_xlabel('Tipo de Contrato')
axes[1,1].legend(['Ativo', 'Churn'])

plt.suptitle('')  # Remove o título automático
plt.tight_layout()
plt.show()

print("\n💡 Observe os padrões nos gráficos acima!")

## 🔧 Passo 4: Preparar Dados para ML

Machine Learning não trabalha com texto! Precisamos converter variáveis categóricas em números.

In [ ]:
# Criar cópia do dataset
df_ml = df.copy()

# Converter Tipo_Contrato para números
# Mensal=0, Anual=1, Bienal=2
le = LabelEncoder()
df_ml['Tipo_Contrato'] = le.fit_transform(df_ml['Tipo_Contrato'])

print("✅ Conversão realizada!")
print("\nMapeamento:")
for i, tipo in enumerate(le.classes_):
    print(f"  {tipo} → {i}")

# Separar features (X) e target (y)
X = df_ml.drop('Churn', axis=1)  # Features: todas exceto Churn
y = df_ml['Churn']                # Target: apenas Churn

print(f"\n📊 Shape dos dados:")
print(f"   Features (X): {X.shape}")
print(f"   Target (y): {y.shape}")

## ✂️ Passo 5: Dividir em Treino e Teste

**Por que dividir?**
- 🎓 **Treino (70%)**: O modelo aprende com esses dados
- 🧪 **Teste (30%)**: Avaliamos o modelo com dados que ele nunca viu
- 🎯 Assim sabemos se o modelo realmente funciona!

In [ ]:
# Dividir dados: 70% treino, 30% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✅ Dados divididos!")
print(f"\n📊 Conjunto de TREINO:")
print(f"   Tamanho: {len(X_train)} clientes ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Churn: {y_train.sum()} casos ({y_train.mean()*100:.1f}%)")

print(f"\n📊 Conjunto de TESTE:")
print(f"   Tamanho: {len(X_test)} clientes ({len(X_test)/len(X)*100:.0f}%)")
print(f"   Churn: {y_test.sum()} casos ({y_test.mean()*100:.1f}%)")

## 🌳 Passo 6: Criar e Treinar o Modelo

Vamos usar uma **Árvore de Decisão** - um algoritmo intuitivo que funciona como um fluxograma de decisões!

In [ ]:
# Criar o modelo
modelo = DecisionTreeClassifier(max_depth=5, random_state=42)

print("🌱 Modelo criado!")
print("📚 Agora vamos treinar...\n")

# Treinar o modelo
modelo.fit(X_train, y_train)

print("✅ Treinamento concluído!")
print("🎓 O modelo aprendeu padrões dos dados de treino!")

## 🎯 Passo 7: Fazer Previsões

In [ ]:
# Fazer previsões nos dados de teste
y_pred = modelo.predict(X_test)

print("🔮 Previsões realizadas!")
print(f"\nExemplo das 10 primeiras previsões:")
print("Real   | Previsto")
print("-" * 20)
for real, prev in list(zip(y_test[:10], y_pred[:10])):
    status = "✅" if real == prev else "❌"
    print(f"  {real}    |    {prev}     {status}")

## 📊 Passo 8: Avaliar Performance

### Métrica 1: Acurácia
Porcentagem de previsões corretas

In [ ]:
# Calcular acurácia
acuracia_treino = accuracy_score(y_train, modelo.predict(X_train))
acuracia_teste = accuracy_score(y_test, y_pred)

print("🎯 ACURÁCIA DO MODELO")
print("=" * 40)
print(f"Treino: {acuracia_treino*100:.2f}%")
print(f"Teste:  {acuracia_teste*100:.2f}%")
print("\n💡 Quanto maior, melhor! (100% = perfeito)")

if abs(acuracia_treino - acuracia_teste) < 0.05:
    print("✅ Modelo generaliza bem!")
else:
    print("⚠️ Atenção: pode estar com overfitting")

### Métrica 2: Matriz de Confusão
Mostra onde o modelo acerta e erra

In [ ]:
# Calcular matriz de confusão
cm = confusion_matrix(y_test, y_pred)

# Visualizar
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Ativo', 'Churn'],
            yticklabels=['Ativo', 'Churn'])
plt.title('Matriz de Confusão', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Valor Real', fontsize=12)
plt.xlabel('Valor Previsto', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 Interpretação:")
print(f"✅ Verdadeiros Negativos (TN): {cm[0,0]} - Previu Ativo e estava certo")
print(f"❌ Falsos Positivos (FP): {cm[0,1]} - Previu Churn mas era Ativo")
print(f"❌ Falsos Negativos (FN): {cm[1,0]} - Previu Ativo mas era Churn")
print(f"✅ Verdadeiros Positivos (TP): {cm[1,1]} - Previu Churn e estava certo")

### Métrica 3: Relatório Detalhado

In [ ]:
print("📈 RELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Ativo', 'Churn']))

print("\n💡 Glossário:")
print("  • Precision: Das previsões de Churn, quantas estavam certas?")
print("  • Recall: Dos casos reais de Churn, quantos conseguimos identificar?")
print("  • F1-Score: Média harmônica entre precision e recall")

## 🔍 Passo 9: Interpretar o Modelo

Quais variáveis são mais importantes para prever churn?

In [ ]:
# Importância das features
importancia = pd.DataFrame({
    'Feature': X.columns,
    'Importancia': modelo.feature_importances_
}).sort_values('Importancia', ascending=False)

# Visualizar
plt.figure(figsize=(10, 6))
plt.barh(importancia['Feature'], importancia['Importancia'], color='steelblue')
plt.xlabel('Importância', fontsize=12)
plt.title('Importância das Variáveis para Prever Churn', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n📊 Ranking de Importância:")
for idx, row in importancia.iterrows():
    print(f"{row['Feature']:.<30} {row['Importancia']:.3f}")

## 🚀 Passo 10: Usar o Modelo em Produção

Vamos simular o uso do modelo para novos clientes!

In [ ]:
# Simular 5 novos clientes
novos_clientes = pd.DataFrame({
    'Tempo_Cliente_Meses': [3, 24, 48, 6, 36],
    'Idade': [25, 45, 55, 30, 40],
    'Gasto_Mensal': [45.0, 120.0, 180.0, 50.0, 150.0],
    'Num_Servicos': [1, 3, 5, 2, 4],
    'Chamadas_Suporte': [8, 2, 0, 7, 1],
    'Tipo_Contrato': [0, 1, 2, 0, 1]  # 0=Mensal, 1=Anual, 2=Bienal
})

# Fazer previsões
previsoes = modelo.predict(novos_clientes)
probabilidades = modelo.predict_proba(novos_clientes)

print("🔮 PREVISÕES PARA NOVOS CLIENTES")
print("=" * 60)

for i in range(len(novos_clientes)):
    print(f"\n👤 Cliente {i+1}:")
    print(f"   Tempo como cliente: {novos_clientes.iloc[i]['Tempo_Cliente_Meses']} meses")
    print(f"   Gasto mensal: R$ {novos_clientes.iloc[i]['Gasto_Mensal']:.2f}")
    print(f"   Chamadas suporte: {novos_clientes.iloc[i]['Chamadas_Suporte']}")
    
    prob_churn = probabilidades[i][1] * 100
    if previsoes[i] == 1:
        print(f"   ⚠️  RISCO DE CHURN! ({prob_churn:.1f}% de probabilidade)")
        print(f"   💡 Ação: Entrar em contato e oferecer benefícios")
    else:
        print(f"   ✅ Cliente provavelmente ficará ({100-prob_churn:.1f}% seguro)")

## 🎯 Desafio Extra (Opcional)

### Nível Intermediário:
1. Teste outros algoritmos: RandomForest, LogisticRegression
2. Ajuste os hiperparâmetros do modelo (max_depth, min_samples_split)
3. Crie novas features combinando as existentes

### Nível Avançado:
1. Faça validação cruzada (cross-validation)
2. Trate o desbalanceamento de classes
3. Crie um sistema de pontuação de risco (0-100)

In [ ]:
# 💡 Espaço para seus experimentos!



## 📝 Conclusões e Próximos Passos

### 🎯 O que você aprendeu:

1. ✅ **Preparar dados** para Machine Learning
2. ✅ **Treinar um modelo** de classificação
3. ✅ **Avaliar performance** com múltiplas métricas
4. ✅ **Interpretar resultados** e importância de features
5. ✅ **Aplicar o modelo** em casos reais

### 💼 Aplicações no Mundo Real:

- 🏦 **Bancos**: Prever inadimplência
- 🏥 **Saúde**: Diagnóstico de doenças
- 📧 **Email**: Filtrar spam
- 🛒 **E-commerce**: Recomendar produtos
- 🚗 **Seguros**: Calcular risco de acidentes

### 📈 Como Melhorar o Modelo:

1. **Mais dados**: Quanto mais exemplos, melhor o aprendizado
2. **Melhores features**: Criar variáveis mais informativas
3. **Feature engineering**: Combinar e transformar variáveis
4. **Experimentar algoritmos**: Testar diferentes modelos
5. **Otimização**: Ajustar hiperparâmetros

---

## 🎉 Parabéns!

Você acabou de criar, treinar e avaliar seu **primeiro modelo de Machine Learning**!

### Próximo passo:
No **Lab 3**, vamos explorar **Deep Learning** e criar uma **rede neural** para reconhecer dígitos escritos à mão!

---

**📚 Recursos Adicionais:**
- Scikit-learn Tutorial: https://scikit-learn.org/stable/tutorial/
- Kaggle Learn: https://www.kaggle.com/learn
- Machine Learning Mastery: https://machinelearningmastery.com/